In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tongpython/cat-and-dog")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/cat-and-dog


In [7]:
import numpy as np 
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [20]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)

In [21]:
train_data = train_gen.flow_from_directory(
    "/kaggle/input/cat-and-dog/training_set/training_set",
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"
)

val_data = val_gen.flow_from_directory(
    "/kaggle/input/cat-and-dog/test_set/test_set",
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"
)


Found 8005 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.


In [23]:
base_model=VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

I0000 00:00:1767727001.080599      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [24]:
for layer in base_model.layers:
    layer.trainable=False

In [25]:
model=Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid'),
])

In [26]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [27]:
history=model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1767727252.460532     977 service.cc:148] XLA service 0x7b890c00e080 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767727252.461336     977 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767727252.866114     977 cuda_dnn.cc:529] Loaded cuDNN version 90300


  2/251 ━━━━━━━━━━━━━━━━━━━━ 18s 76ms/step - accuracy: 0.4219 - loss: 1.9203 

I0000 00:00:1767727261.106703     977 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


251/251 ━━━━━━━━━━━━━━━━━━━━ 152s 564ms/step - accuracy: 0.7698 - loss: 0.7241 - val_accuracy: 0.9061 - val_loss: 0.2198
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 93s 369ms/step - accuracy: 0.8724 - loss: 0.2949 - val_accuracy: 0.9140 - val_loss: 0.1983
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 93s 369ms/step - accuracy: 0.8901 - loss: 0.2554 - val_accuracy: 0.9174 - val_loss: 0.1871
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 94s 374ms/step - accuracy: 0.8968 - loss: 0.2479 - val_accuracy: 0.9179 - val_loss: 0.1927
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 92s 365ms/step - accuracy: 0.9089 - loss: 0.2276 - val_accuracy: 0.9239 - val_loss: 0.1893
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 93s 369ms/step - accuracy: 0.9091 - loss: 0.2228 - val_accuracy: 0.9303 - val_loss: 0.1774
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 95s 378ms/step - accuracy: 0.9011 - loss: 0.2200 - val_accuracy: 0.9348 - val_loss: 0.1688
Epoch 8/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 91s 363ms/step - accuracy: 0.9206 - loss: 0.1988 - va

In [31]:
from tensorflow.keras.preprocessing import image

In [33]:
img=image.load_img('/kaggle/input/cat-and-dog/test_set/test_set/cats/cat.4004.jpg', target_size=(224,224))
img_arr=image.img_to_array(img)/255.0
img_arr=np.expand_dims(img_arr, axis=0)
prediction=model.predict(img_arr)

print('dog' if prediction[0][0]>0.5 else 'cat')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
cat


In [ ]:
model